# PopGMM — probabilistic ancestry inference and population-stratification control

Projects a CTEPH case / AGP3K control cohort onto the BioBank Japan (BBJ) PCA
reference space, models the reference with a Gaussian mixture, and emits
ancestry-homogeneous `FID IID` keep-lists for downstream association analysis.

**Run it top to bottom.** Cells are a linear dependency chain; nothing needs to
be executed out of order.

| | |
|---|---|
| Inputs | `data/bbj.pca_base.eigenval`, `data/*.sscore` (PLINK2 `--score` output) |
| Deliverable | `results/09_threshold_sample_exports/retained_all_thr_*.fid_iid.txt` |
| Parameters | all in [`scripts/params.py`](scripts/params.py) — do not hard-code them here |
| Verification | `python -m tools.verify_results --baseline results --candidate results_verify` |
| Reproducibility evidence | [`docs/reproducibility_probe.md`](docs/reproducibility_probe.md) |

In [ ]:
import dataclasses
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scripts.params as params
from scripts.artifacts import ArtifactCache, run_environment

plt.style.use("default")

# "fresh" executes every step and writes all of its output files; this is the
# mode for publication and verification runs, and it is the default.
# "resume" reuses cached STEP0-STEP2 artifacts to skip the ~5 minute GMM search
# while iterating -- but a cache hit means those steps' tables, figures and
# audit logs are NOT rewritten, so never verify a "resume" run.
RUN_MODE = "fresh"

cache = ArtifactCache(mode=RUN_MODE)

params.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
(params.RESULTS_ROOT / "run_environment.json").write_text(
    json.dumps(run_environment(RUN_MODE), indent=2, sort_keys=True) + "\n"
)

print(f"results root : {params.RESULTS_ROOT}")
print(f"run mode     : {RUN_MODE}")

## STEP0 — Data loading

Reads the shared `.sscore` matrix once and splits it by `IID` prefix into the
BBJ reference cohort (`bbj_*`, n = 183,013) and the study cohort (n = 3,571),
deriving case/control IID lists from `PHENO1` (2 = case, 1 = control).

**Outputs (in memory):** `eigenval`, `bbj_samples`, `our_samples`,
`our_case_iids`, `our_ctrl_iids`.

In [ ]:
from scripts.data_loading import DataLoadingConfig, load_step0_bbj_and_our

config_step0 = DataLoadingConfig(
    chunksize=50000,
    bbj_prefix="bbj_",
    verbose=True,
    phenotype_column="PHENO1",
    case_value=2,
    control_value=1,
)

eigenval, bbj_samples, our_samples, our_case_iids, our_ctrl_iids = cache.compute(
    "step0_data_loading",
    lambda: load_step0_bbj_and_our(
        eigenval_path=params.EIGENVAL_PATH,
        sscore_path=params.SSCORE_PATH,
        config=config_step0,
    ),
    config=config_step0,
    files=[params.EIGENVAL_PATH, params.SSCORE_PATH],
    writes_side_effects=False,
)

## STEP1 — HDBSCAN denoising of the BBJ reference

Removes sparse outliers in PC1–PC2 space so the mixture model is fitted to
stable population structure rather than to scattered noise.

**Note:** `hdbscan_filtering` prefers python-hdbscan and silently falls back to
`sklearn.cluster.HDBSCAN` if it is missing — the two disagree on the noise set,
so the pin in `requirements.txt` is load-bearing.

**Outputs:** `results/01_hdbscan_filtering/` — denoised sample table, summary
JSON, 3-panel QC figure. Expected: 183,013 → 181,817 (1,196 noise, 0.65 %).

In [ ]:
from scripts.hdbscan_filtering import HDBSCANConfig, run_hdbscan_denoise_bbj

config_step1 = HDBSCANConfig(
    n_pcs_hdbscan=2,
    use_zscale_hdbscan=True,
    min_cluster_size=50,
    min_samples=6,
    cluster_selection_epsilon=0.005,
    cluster_selection_method="eom",
    metric="euclidean",
    alpha=0.8,
    allow_single_cluster=True,
    leaf_size=40,
    algorithm="best",
    approx_min_span_tree=True,
    gen_min_span_tree=False,
    output_dir=params.STEP1_DIR,
    save_plot=True,
    save_tables=True,
    save_full_table=False,
    verbose=True,
)

bbj_hdbscan = cache.compute(
    "step1_hdbscan",
    lambda: run_hdbscan_denoise_bbj(
        bbj_samples=bbj_samples,
        eigenval=eigenval,
        config=config_step1,
    ),
    config=config_step1,
    frames=[bbj_samples],
)

# Main downstream input: only non-noise BBJ samples.
bbj_samples_filtered = bbj_hdbscan.bbj_samples_filtered.drop(
    columns=["HDBSCAN_Label"], errors="ignore"
)

## STEP2 — GMM clustering of the BBJ reference

Fits full-covariance Gaussian mixtures for `k = 2..100` on the fixed 2-PC
embedding and selects the minimum-BIC model that has no empty cluster.

This is the expensive step (~5 min across 6 processes) and the fitted model
exists nowhere else, which is why it is cached.

**Outputs:** `results/02_gmm_clustering/` — clustered sample table, BIC search
table, cluster summary, overview figure, and `tmp/` convergence and BIC audit
logs. Expected: `best_k = 26`, `BIC = -2,682,580.33`.

**On reproducibility:** 15 of the 99 candidate fits vary by up to 1.39 BIC units
between runs (one float32 ULP of the log-likelihood amplified by the sample
count). The margin to the runner-up is 155.92 units, so `best_k` does not move.
See `docs/reproducibility_probe.md`.

In [ ]:
from scripts.gmm_clustering import GMMConfig, run_gmm_fixed_pcs

config_step2 = GMMConfig(
    fixed_n_pcs=2,
    k_min=2,
    k_max=100,
    use_zscale=False,
    covariance_type="full",
    n_init=3,
    init_params="kmeans",
    reg_covar=1e-6,
    max_iter=200,
    random_state=params.RANDOM_SEED,
    search_max_samples=200000,
    search_workers=6,
    require_non_empty_clusters=True,
    output_dir=params.STEP2_DIR,
    save_plot=True,
    save_tables=True,
    verbose=True,
)

gmm_result = cache.compute(
    "step2_gmm",
    lambda: run_gmm_fixed_pcs(
        bbj_samples_filtered=bbj_samples_filtered,
        eigenval=eigenval,
        config=config_step2,
    ),
    config=config_step2,
    upstream=["step1_hdbscan"],
    frames=[bbj_samples_filtered],
)

bbj_samples_gmm = gmm_result.bbj_samples_with_cluster
gmm_summary = gmm_result.summary
gmm_model = gmm_result.model  # needed by STEP3, STEP3_tmp, STEP4, STEP4_tmp, STEP5

## STEP3 — Merge nearby GMM components

Computes pairwise Mahalanobis distances between component means using the
pooled covariance `S_ij = 0.5 (Σ_i + Σ_j)`, clusters them hierarchically, and
cuts the dendrogram at `params.MERGE_THRESHOLD_MAIN`.

The "mainland" merged cluster is the one containing the most pre-merge
components (ties broken by smallest id).

**Outputs:** `results/03_gmm_component_merging/` — merged sample table,
posterior `.npy`, merge map, Mahalanobis distances, mainland reference,
2×2 overview figure. Expected: 26 → 6 clusters, mainland = 17 components.

In [ ]:
from scripts.gmm_component_merging import (
    GMMComponentMergingConfig,
    run_gmm_component_merging,
)

config_step3 = GMMComponentMergingConfig(
    merge_threshold=params.MERGE_THRESHOLD_MAIN,
    linkage_method="average",
    output_dir=params.STEP3_DIR,
    save_plot=True,
    save_tables=True,
    # Panel D: stable and interpretable legend range across runs
    conf_scale_mode="fixed",
    conf_scale_fixed_vmin=0.95,
    conf_scale_fixed_vmax=1.00,
    # Compress differences near 1.0 (high-confidence end)
    conf_norm="power",
    conf_power_gamma=0.40,
    verbose=True,
)

merge_result = run_gmm_component_merging(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_step3,
)

merge_map = merge_result.merge_map
mainland_premerge_cluster_ids = merge_result.mainland_premerge_cluster_ids

## STEP3_tmp — Merge-threshold sensitivity

Re-runs STEP3 at each alternative threshold in
`params.MERGE_THRESHOLD_SENSITIVITY` (currently `{2.5: "STEP3_tmp"}`) and stores
the result in its own subdirectory. Does not affect the main analysis.

`dataclasses.replace` derives each config from `config_step3`, which *guarantees*
every unlisted field — in particular the four visualization settings — stays
identical to the main run. The previous copy-pasted cell only asserted that in a
comment.

In [ ]:
merge_results_sensitivity = {}

for _threshold, _subdir in params.MERGE_THRESHOLD_SENSITIVITY.items():
    _config = dataclasses.replace(
        config_step3,
        merge_threshold=_threshold,
        output_dir=params.merge_threshold_dir(_subdir),
    )
    merge_results_sensitivity[_threshold] = run_gmm_component_merging(
        gmm_model=gmm_model,
        bbj_samples_gmm=bbj_samples_gmm,
        eigenval=eigenval,
        gmm_summary=gmm_summary,
        config=_config,
    )

## STEP4 — Assign the study cohort to the pre-merge GMM components

Projects the study samples into the reference mixture and takes
`predict_proba`, using an **identity** label map so each original component
stays separate. `Assignment_Confidence` is the maximum posterior.

The second half compares case vs control distributions across all 20 PCs within
the mainland cluster (Welch *t* + Mann-Whitney, BH-FDR) and exports the mainland
sample list.

**Outputs:** `results/04_our_assignment/` — posterior table, assignment figure,
20-PC KDE panel, `mainland_samples.fid_iid.txt`. Expected: 3,099 / 3,571
mainland (434 cases, 2,665 controls).

In [ ]:
from typing import Any, cast

from scripts.our_assignment import OURAssignmentConfig, run_our_assignment_to_merged_gmm

# Identity map: each original GMM component maps to itself (pre-merge assignment).
n_components_premerge = int(getattr(cast(Any, gmm_model), "n_components"))
premerge_label_map = {int(k): int(k) for k in range(n_components_premerge)}

config_step4 = OURAssignmentConfig(
    output_dir=params.STEP4_DIR,
    save_plot=True,
    save_tables=True,
    output_file="our_posterior_probabilities_premerge.tsv",
    figure_file="our_assignment_premerge.png",
    case_label=params.CASE_LABEL,
    control_label=params.CONTROL_LABEL,
    bbj_alpha=0.20,
    verbose=True,
)

step4_out = run_our_assignment_to_merged_gmm(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    label_map=premerge_label_map,
    merge_map=merge_map,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    training_use_zscale=config_step2.use_zscale,
    config=config_step4,
)

# Mainland-only all-PC KDE visualization and sample export.
from scripts.mainland_all_pcs_kde import MainlandAllPCsKDEConfig, run_mainland_all_pcs_kde

config_step4_mainland = MainlandAllPCsKDEConfig(
    output_dir=params.STEP4_DIR,
    save_plot=True,
    save_tables=True,
    output_file="mainland_samples.fid_iid.txt",
    figure_file="mainland_all_pcs_kde.png",
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    bbj_color="#1F78B4",
    case_color="#E31A1C",
    alpha=0.65,
    verbose=True,
)

mainland_kde_out = run_mainland_all_pcs_kde(
    df_results=step4_out.df_results,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    mainland_cluster_ids=mainland_premerge_cluster_ids,
    eigenval=eigenval,
    step4_config=config_step4,
    config=config_step4_mainland,
)

## STEP4_tmp — Mainland rank-cumulative analysis (Rank 1 → 17)

Ranks the 17 mainland components by case/control ratio using direct pre-merge
MAP counts. Then, for each `k = 1..17`, merges the top-k into one group,
recomputes and renormalizes the posteriors, reassigns by argmax, and reports
`GWAS_Neff`, PC1–2 heterogeneity and the Pareto front.

`forced_recommended_rank` overrides the Pareto-auto choice with
`params.MAINLAND_RANK_K`; set it to `None` to use the automatic detection.
**STEP5 reads the cut back off this step's output**, so the value is stated once.

**Outputs:** `results/04_our_assignment/STEP4_tmp/` — rank table, cumulative
metrics, decision table, progression figure (PNG + PDF).

In [ ]:
from scripts.step4_tmp_mainland_rank_progression import (
    Step4TmpMainlandRankProgressionConfig,
    run_step4_tmp_mainland_rank_progression,
)

config_step4_tmp = Step4TmpMainlandRankProgressionConfig(
    output_dir=params.STEP4_TMP_DIR,
    max_rank=params.MAINLAND_MAX_RANK,
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    forced_recommended_rank=params.MAINLAND_RANK_K,  # None -> Pareto-auto detection
    save_plot=True,
    show_plot=False,
    verbose=True,
)

step4_tmp_out = run_step4_tmp_mainland_rank_progression(
    df_results=step4_out.df_results,
    merge_map=merge_map,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    gmm_model=gmm_model,
    gmm_summary=gmm_summary,
    config=config_step4_tmp,
)

step4_tmp_rank_table = step4_tmp_out.rank_table

## STEP5 — Mainland subcluster global posterior reassignment

Takes the top-ranked mainland components from STEP4_tmp, merges them into a
single "Mainland Subcluster" group while every other component stays separate,
renormalizes the posteriors over the resulting 18 groups and reassigns by argmax.

The cut is **read back from STEP4_tmp** rather than retyped — previously `9` was
hard-coded in both cells with nothing keeping them in sync. The assertion makes
a divergence loud instead of silent.

**Outputs:** `results/05_customize_cluster_assignment/` — posterior table,
assignment figure, group summary JSON. Expected: components
{0, 2, 3, 7, 12, 14, 18, 24, 25}.

In [ ]:
from scripts.customize_cluster_assignment import (
    CustomizeClusterAssignmentConfig,
    run_customize_cluster_assignment,
)

# The rank cut comes from STEP4_tmp's own output, not a second literal.
step5_included_rank = int(step4_tmp_out.recommended_rank)
assert step5_included_rank == params.MAINLAND_RANK_K, (
    f"STEP4_tmp recommended rank {step5_included_rank} != "
    f"params.MAINLAND_RANK_K {params.MAINLAND_RANK_K}"
)

_included = set(
    int(v)
    for v in step4_tmp_rank_table.loc[
        step4_tmp_rank_table["Rank"] <= step5_included_rank, "Cluster"
    ]
)
step5_exclude_ids = tuple(
    sorted(set(int(v) for v in mainland_premerge_cluster_ids) - _included)
)

config_step5 = CustomizeClusterAssignmentConfig(
    output_dir=params.STEP5_DIR,
    save_plot=True,
    save_tables=True,
    output_file="our_posterior_probabilities_customize_merged.tsv",
    figure_file="our_assignment_customize_merged.png",
    custom_group_label="Mainland Subcluster",
    exclude_cluster_ids=step5_exclude_ids,
    case_label=config_step4.case_label,
    control_label=config_step4.control_label,
    bbj_alpha=config_step4.bbj_alpha,
    verbose=True,
)

step5_out = run_customize_cluster_assignment(
    gmm_model=gmm_model,
    bbj_samples_gmm=bbj_samples_gmm,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    mainland_premerge_cluster_ids=mainland_premerge_cluster_ids,
    eigenval=eigenval,
    gmm_summary=gmm_summary,
    config=config_step5,
)

df_results_step5 = step5_out.df_results

## STEP6 — Subcluster-only visualization and export

PC1–2 scatter restricted to the Mainland Subcluster, plus the 20-PC case/control
KDE panel for that subset and its unfiltered `FID IID` export.

**Outputs:** `results/06_mainland_subcluster_only/`. Expected: 2,193 / 3,571
samples (411 cases, 1,782 controls).

In [ ]:
from scripts.mainland_subcluster_only import (
    MainlandSubclusterOnlyConfig,
    run_mainland_subcluster_only,
)

config_step6 = MainlandSubclusterOnlyConfig(
    output_dir=params.STEP6_DIR,
    sample_id_file="mainland_subcluster_samples.fid_iid.txt",
    figure_file="mainland_subcluster_only.png",
    mainland_group_label=config_step5.custom_group_label,
    assigned_group_col="Assigned_Mainland_Subcluster_Group",
    confidence_col="Assignment_Confidence",
    case_label=config_step5.case_label,
    control_label=config_step5.control_label,
    mainland_group_color=config_step5.custom_group_color,
    bbj_color=config_step5.bbj_color,
    bbj_alpha=config_step5.bbj_alpha,
    our_point_size=config_step5.our_point_size,
    save_plot=True,
    show_plot=False,
    verbose=True,
)

step6_out = run_mainland_subcluster_only(
    df_results_step5=df_results_step5,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    bbj_samples_gmm=bbj_samples_gmm,
    eigenval=eigenval,
    config=config_step6,
)

# Mainland-subcluster all-PC KDE visualization and sample export.
from scripts.mainland_subcluster_all_pcs_kde import (
    MainlandSubclusterAllPCsKDEConfig,
    run_mainland_subcluster_all_pcs_kde,
)

config_step6_mainland = MainlandSubclusterAllPCsKDEConfig(
    output_dir=params.STEP6_DIR,
    save_plot=True,
    save_tables=True,
    output_file="mainland_subcluster_samples.fid_iid.txt",
    figure_file="mainland_subcluster_all_pcs_kde.png",
    mainland_group_label=config_step5.custom_group_label,
    assigned_group_col="Assigned_Mainland_Subcluster_Group",
    confidence_col="Assignment_Confidence",
    case_label=config_step5.case_label,
    control_label=config_step5.control_label,
    bbj_color="#1F78B4",
    case_color="#E31A1C",
    alpha=0.65,
    verbose=True,
)

mainland_subcluster_kde_out = run_mainland_subcluster_all_pcs_kde(
    df_results_step5=df_results_step5,
    our_samples=our_samples,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step6_mainland,
)

df_mainland_subcluster = step6_out.df_mainland_subcluster

## STEP7 — Confidence distribution

Histogram and CDF of `Assignment_Confidence` within the Mainland Subcluster,
overall and split by case/control.

**Outputs:** `results/07_mainland_subcluster_confidence_distribution/`.

In [ ]:
from scripts.mainland_subcluster_confidence_distribution import (
    MainlandSubclusterConfidenceDistributionConfig,
    run_mainland_subcluster_confidence_distribution,
)

config_step7 = MainlandSubclusterConfidenceDistributionConfig(
    output_dir=params.STEP7_DIR,
    case_label=params.CASE_LABEL,
    control_label=params.CONTROL_LABEL,
    save_plot=True,
    show_plot=False,
    verbose=True,
)

step7_out = run_mainland_subcluster_confidence_distribution(
    df_mainland_subcluster=df_mainland_subcluster,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step7,
)

## STEP8 — Confidence-threshold screening

Retained/removed counts and rates per group at each threshold in
`params.CONFIDENCE_THRESHOLDS`, plus the minimum case confidence
(`include_case_min_threshold=True`, computed inside the module).

`params.CONFIDENCE_THRESHOLDS` is shared with STEP9 — the screening table and
the exported keep-lists must describe the same cutoffs.

**Outputs:** `results/08_mainland_subcluster_confidence_screening/`.

In [ ]:
from scripts.mainland_subcluster_confidence_threshold_screening import (
    MainlandSubclusterConfidenceThresholdScreeningConfig,
    run_mainland_subcluster_confidence_threshold_screening,
)

config_step8 = MainlandSubclusterConfidenceThresholdScreeningConfig(
    output_dir=params.STEP8_DIR,
    fixed_thresholds=params.CONFIDENCE_THRESHOLDS,
    include_case_min_threshold=True,
    case_label=params.CASE_LABEL,
    control_label=params.CONTROL_LABEL,
    save_plot=True,
    show_plot=False,
    verbose=True,
)

step8_out = run_mainland_subcluster_confidence_threshold_screening(
    df_mainland_subcluster=df_mainland_subcluster,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step8,
)

## STEP9 — Threshold-based sample export

Writes the PLINK-ready keep-lists. **This is the deliverable of the whole
pipeline.**

```bash
plink2 --pfile <dataset> \
       --keep results/09_threshold_sample_exports/retained_all_thr_0.9000.fid_iid.txt \
       --make-pgen --out <dataset>.ancestry_qc
```

**Outputs:** `results/09_threshold_sample_exports/` — one
`retained_all_thr_*.fid_iid.txt` per threshold (the five fixed ones plus the
case-minimum, 0.4240) and a retained/removed summary table.

In [ ]:
from scripts.mainland_subcluster_threshold_sample_export import (
    MainlandSubclusterThresholdSampleExportConfig,
    run_mainland_subcluster_threshold_sample_export,
)

config_step9 = MainlandSubclusterThresholdSampleExportConfig(
    output_dir=params.STEP9_DIR,
    summary_file="threshold_retained_removed_summary.tsv",
    fixed_thresholds=params.CONFIDENCE_THRESHOLDS,
    include_case_min_threshold=True,
    case_label=params.CASE_LABEL,
    control_label=params.CONTROL_LABEL,
    fid_col="FID",
    iid_col="IID",
    confidence_col="Assignment_Confidence",
    export_case_ctrl_files=False,
    verbose=True,
)

step9_out = run_mainland_subcluster_threshold_sample_export(
    df_mainland_subcluster=df_mainland_subcluster,
    our_case_iids=our_case_iids,
    our_ctrl_iids=our_ctrl_iids,
    config=config_step9,
)

## Provenance — configuration snapshot

Serializes every step config to `<RESULTS_ROOT>/run_config_snapshot.json`.
Diffing two snapshots proves a refactor did not alter any parameter *without
re-running the pipeline*, which makes it the cheap pre-flight check before
spending 7 minutes on a full verification run.

In [ ]:
_config_snapshot = {
    name: dataclasses.asdict(obj)
    for name, obj in sorted(globals().items())
    if name.startswith("config_step") and dataclasses.is_dataclass(obj)
}
_config_snapshot["_derived"] = {
    "step5_included_rank": step5_included_rank,
    "step5_exclude_cluster_ids": list(step5_exclude_ids),
    "mainland_premerge_cluster_ids": [int(v) for v in mainland_premerge_cluster_ids],
    "merge_threshold_sensitivity": {
        str(k): v for k, v in params.MERGE_THRESHOLD_SENSITIVITY.items()
    },
}

_snapshot_path = params.RESULTS_ROOT / "run_config_snapshot.json"
_snapshot_path.write_text(
    json.dumps(_config_snapshot, indent=2, sort_keys=True, default=str) + "\n"
)
print(f"wrote {len(_config_snapshot) - 1} step configs -> {_snapshot_path}")